# BAYES-LOSVD workflow

This notebook runs the full pipeline on the example data set **NGC0000** (a MUSE cube shipped in `data/`):

| Step | Script | Output |
|---|---|---|
| 1. Configure | `config_files/*.properties` | – |
| 2. Preprocess | `bayes_losvd_preproc_data.py` | `preproc_data/<run>.hdf5` (+ `.pdf` with diagnostics) |
| 3. Fit | `bayes_losvd_run.py` | `results/<run>/<run>_results.hdf5` |
| 4. Check convergence | `bayes_losvd_check_results.py` | R̂ / ESS report |
| 5. Inspect fits | `losvd_explorer/` | interactive viewer, PNG/PDF per bin |
| 6. Use the results | `bayes_losvd_load_hdf5.py` | arrays for your analysis |

Run the notebook from the `scripts/` directory (all scripts use paths relative to it), with Jupyter started from
the activated BAYES-LOSVD environment. Step 5 additionally needs
`conda install -c conda-forge plotly ipywidgets anywidget voila`.

Cells starting with `!` are ordinary shell commands: drop the `!` and the same line works in a terminal
(from `scripts/`).

**Before running:** this full copy includes existing local results. Use a fresh
preprocessing section/run name (and the corresponding paths) to avoid replacing
them. The unchanged runner does not append GP/SP to filenames or preserve prior
packed bins on a partial rerun. The checker alone does not verify completeness,
NaN diagnostics or divergences. These examples are corrected instructions,
not executed notebook outputs. See the manual's Current implementation page.


## 1. Configure

Each run is a section in a TOML file; the section name becomes the run name (`NGC0000` below, so all outputs are
called `NGC0000…`). The cell below writes the example to `config_files/NGC0000.properties`; for your own data add
a section to `config_files/preproc.properties` (see `example_preproc.properties` for more examples).

| Key | Meaning |
|---|---|
| `filename` | input file in `data/` |
| `instrument` | entry in `instruments.properties`: data reader and line-spread function |
| `redshift`, `lmin`, `lmax` | redshift and rest-frame wavelength range [Å] |
| `vmax`, `velscale` | LOSVD grid within −vmax to +vmax (use saved `xvel` for exact endpoints) in steps of `velscale` km/s (also the spectral pixel size) |
| `snr`, `snr_min` | target S/N for spatial binning (`0` = no binning); spaxels fainter than the S/N ≈ `snr_min` isophote are dropped |
| `template_lib`, `npca` | template library in `templates/`; number of PCA components (`0` = use the templates directly) |
| `xcen`, `ycen` | optional galaxy centre in pixels |

In [ ]:
%%writefile ../config_files/NGC0000.properties
[NGC0000]
filename     = "NGC0000.fits"
instrument   = "MUSE-WFM"
redshift     = 0.008764
lmin         = 4750.0
lmax         = 5500.0
vmax         = 700.0
velscale     = 60.0
snr          = 50.0
snr_min      = 3.0
template_lib = "MILES_SSP"
npca         = 5

## 2. Preprocess

`bayes_losvd_preproc_data.py` processes every section of the config file:

- **data:** shift to the rest frame, cut to `lmin…lmax`, drop faint spaxels, bin to the target S/N,
  log-rebin to `velscale` and normalise each spectrum to a mean of 1;
- **templates:** load the library, optionally reduce it to `npca` PCA components, convolve to the instrumental
  resolution of the data and resample to the same wavelength grid.

The result is `preproc_data/<run>.hdf5` plus a PDF with flux and S/N maps.

In [ ]:
!python bayes_losvd_preproc_data.py -c ../config_files/NGC0000.properties

## 3. Fit

For every bin, `bayes_losvd_run.py` samples the model

$$\text{spectrum} = (\text{templates} \ast \text{LOSVD}) \times \text{Legendre continuum}$$

with NumPyro/NUTS. The LOSVD is a normalised histogram on the `xvel` grid with a Gaussian-process prior (`GP`,
hyper-parameters `ell_gp`, `sigma_gp`) or a Dirichlet prior (`SP`). The likelihood uses a constant noise of
1/min(S/N, 100) per bin. Afterwards a Gauss-Hermite profile (v, σ, h3, h4) is fitted to every LOSVD sample.

| Flag | Meaning | Default |
|---|---|---|
| `-f` | preprocessed file | – |
| `-l` | bins: `all`, a list `0,5,12`, `odd` (= 0, 2, 4, …) or `even` (= 1, 3, 5, …) | `all` |
| `-m` | mask file with spectral regions to exclude | none |
| `-p` | order of the Legendre continuum (`0` = none) | 5 |
| `-n`, `-c` | samples (and warm-up steps) per chain, number of chains | 500, 2 |
| `-j` | bins fitted in parallel | 1 |
| `-t` | model: `GP` or `SP` | GP |
| `-s` | also save the MCMC chains (NetCDF) | off |
| `--extra_pars` | fix prior parameters, e.g. `ell_gp=3.0,sigma_gp=0.5` | – |

Each bin writes a `*_summary_bin<N>.txt` (used in step 4); at the end all bins are packed into
`results/<run>/<run>_results.hdf5`. Bins that were not fitted are stored as NaN.

The demo below fits only bins 0–3 (runtime depends on hardware and first-run compilation). For the full map use `-l all` with more parallel jobs,
e.g. `python bayes_losvd_run.py -f ../preproc_data/NGC0000.hdf5 -m ../config_files/emission_lines.mask -l all -j 8`.

In [ ]:
!python bayes_losvd_run.py -f ../preproc_data/NGC0000.hdf5 -m ../config_files/emission_lines.mask -t GP -l 0,1,2,3 -j 4

## 4. Check convergence

Flags every variable with R̂ > 1.1 or an effective sample size below 100 (thresholds: `-r`, `-e`). For a new run, increase samples or chains if needed. **Do not partially rerun an existing packed result:** packing replaces it without preserving earlier bins. Use a new preprocessing/run name and retain the old output directory.

In [ ]:
!python bayes_losvd_check_results.py -d ../results/NGC0000

## 5. Inspect the fits

The `losvd_explorer` directory replaces `bayes_losvd_inspect_fits.py`. For one bin it shows the map
position, the LOSVD with 68 % / 99.8 % bands and the Gauss-Hermite profile, the template weights, and the spectral
fit with residuals. 

**Python API:**

In [ ]:
%matplotlib inline
import sys

sys.path.insert(0, "losvd_explorer")
from losvd_data import load_dataset
import losvd_static

RESULTS = "../results/NGC0000/NGC0000_results.hdf5"
ds = load_dataset(RESULTS)
losvd_static.render_bin_figure(ds, ds.center_bin)

**Interactive viewer.** Click a bin on the map to inspect it; switch the map between v, σ, h3, h4, errors, S/N and
χ²; save figures as PNG/PDF with *Save bin* / *Save all*.

In the browser: run this in a terminal from `scripts/` (it starts a local server and opens the viewer at
the address printed by Voila; stop it with `Ctrl+C`):

```bash
losvd_explorer/start_explorer.sh ../results/NGC0000
```

Or directly inside this notebook:

In [ ]:
from losvd_explorer_app import launch_app

app = launch_app(RESULTS)

**Command line** (replaces `bayes_losvd_inspect_fits.py`; figures are saved next to the results file as
`<file stem>_bin<N>.png`). Other options: `-l 12` for one bin, `-l 0,5,10-20` for a selection, `--format pdf`,
`--map sigma`, `--outdir figs/`; without `-s` the figure is shown instead of saved.

In [ ]:
!python losvd_explorer/inspect_fits.py -f ../results/NGC0000/NGC0000_results.hdf5 -l all -s

## 6. Use the results

The results file has an `in/` group (preprocessed input, fit mask) and an `out/` group. Each `out/` array has the
shape `(nbins, 7, …)`, where axis 1 holds:

| 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| 0.1 % | 15.9 % | median | 84.1 % | 99.9 % | mean | std |

Main quantities: `losvd`, `vel_star`, `sigma_star`, `h3_star`, `h4_star`, `snr_real`, `model_spec`, `continuum`,
`weights`, `ell_gp`, `sigma_gp`.

In [ ]:
from bayes_losvd_load_hdf5 import load_hdf5

res = load_hdf5(RESULTS, verbose=False)
vel = res["vel_star"][:, 2]                                         # median velocity per bin
vel_err = 0.5 * (res["vel_star"][:, 3] - res["vel_star"][:, 1])     # 1σ error
print(res["losvd"].shape, vel[:4], vel_err[:4])

## Notes

- `-o` only changes where the individual bins are written; packing always reads from `../results/`.
- Unfinished runs may leave per-bin files. Explorer scanning expects a base `*_results.hdf5`; standalone per-bin discovery/loading is not guaranteed.